# PA3 — treinos pesados no Colab (GPU)

Roda no Colab só o que é caro em CPU: o **MegaDetector** (tamanho dos animais),
os **6 alunos da Parte 1** e todas as rodadas da **Parte 2** (curva de orçamento).
O professor (BioCLIP 2) já rodou localmente: os pseudo-rótulos vêm junto do
repositório (`outputs/part1/pseudo_labels/`).

**Antes de começar**
1. `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)`.
2. Suba o `pa3_dados.zip` (415 MB) para o seu Google Drive em `MeuDrive/PA3/pa3_dados.zip`.
3. `Ambiente de execução → Executar tudo`.

**Se o Colab desconectar:** rode tudo de novo. Checkpoints (a cada época) e
resultados ficam em `MeuDrive/PA3/run/`, e cada script continua de onde parou.

**No fim:** a última célula gera `MeuDrive/PA3/run/outputs_colab.zip`. Baixe e
coloque na pasta do PA3 local (eu cuido do resto: figuras, análise e commit).

In [ ]:
# 1. GPU disponível?
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "Sem GPU: Ambiente de execução -> Alterar o tipo -> GPU"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Google Drive (dados de entrada e persistência dos resultados)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/PA3'
RUN = f'{DRIVE}/run'
import os
assert os.path.exists(f'{DRIVE}/pa3_dados.zip'), f'Suba o pa3_dados.zip para {DRIVE}/'
os.makedirs(f'{RUN}/outputs', exist_ok=True); os.makedirs(f'{RUN}/checkpoints', exist_ok=True)

In [ ]:
# 3. Código: clona (ou atualiza) o repositório
REPO = '/content/PA3_Deep_learning'
if not os.path.exists(REPO):
    !git clone -q https://github.com/matheuspinto4/PA3_Deep_learning.git {REPO}
else:
    !cd {REPO} && git fetch -q && git checkout -q -- . && git pull -q
%cd {REPO}
!git log --oneline -1

In [ ]:
# 4. Dados: descompacta no disco local do Colab (bem mais rápido que ler do Drive)
if not os.path.exists(f'{REPO}/pa3_dados/pa3_dados/LEIAME.md'):
    !mkdir -p {REPO}/pa3_dados && unzip -q -o {DRIVE}/pa3_dados.zip -d {REPO}/pa3_dados
    # o zip pode ou não ter a pasta pa3_dados/ no topo: normaliza para pa3_dados/pa3_dados/
    if not os.path.exists(f'{REPO}/pa3_dados/pa3_dados/LEIAME.md'):
        !mkdir -p /tmp/pa3x && mv {REPO}/pa3_dados/* /tmp/pa3x/ && mkdir -p {REPO}/pa3_dados/pa3_dados && mv /tmp/pa3x/* {REPO}/pa3_dados/pa3_dados/
!ls {REPO}/pa3_dados/pa3_dados/images && ls {REPO}/pa3_dados/pa3_dados/images/train | wc -l

In [ ]:
# 5. Persistência: outputs/ e checkpoints/ do repo passam a apontar para o Drive.
#    Na 1a vez, copia para o Drive os resultados já versionados (Partes 0 e 1);
#    -n = não sobrescreve o que já existe lá (resultados de rodadas anteriores).
import pathlib
if not os.path.islink(f'{REPO}/outputs'):
    !cp -rn {REPO}/outputs/. {RUN}/outputs/
    !rm -rf {REPO}/outputs && ln -s {RUN}/outputs {REPO}/outputs
if not os.path.islink(f'{REPO}/checkpoints'):
    !rm -rf {REPO}/checkpoints && ln -s {RUN}/checkpoints {REPO}/checkpoints
!ls -la {REPO} | grep -E 'outputs|checkpoints'

In [ ]:
# 6. Dependências extras (o resto já vem no Colab) + cache de imagens 224x320 (~2-4 min)
!pip install -q ultralytics
!python scripts/build_cache.py

In [ ]:
# 7. MegaDetector V6 no test_public e no train: tamanho dos animais (~5 min)
!python -u scripts/part1_megadetector.py

In [ ]:
# 8. Parte 1: os 6 alunos (orçamento 5, seed 0), escolha pelo val, figuras (~30-45 min)
!python -u scripts/part1_student.py 2>&1 | tee outputs/part1/student_log_colab.txt

In [ ]:
# 9. Parte 2: piso e aluno escolhido nos orçamentos 1/5/20 (3 seeds no 5) (~30-40 min)
!python -u scripts/part2_budget.py 2>&1 | tee outputs/part2_log_colab.txt

In [ ]:
# 10. Empacota os resultados (sem checkpoints, que são grandes) para baixar
!cd {RUN} && rm -f outputs_colab.zip && zip -qr outputs_colab.zip outputs && ls -la outputs_colab.zip
from google.colab import files
files.download(f'{RUN}/outputs_colab.zip')